In [1]:
import os
import re
import statistics

# --- Configuration ---
# Update these lists based on your specific needs
CKPT_NAME = "Qwen2.5-1.5B" # Change to "gpt2-medium" as needed
APPROACHES = ["distill_lens", "kd", "sft"] #["hybrid_kd", "kd", "logit_lens", "mse", "fdd"]
TYPES = ["fkl", "rkl", "fkl+rkl", "akl", "jsd", "lm"] #["fkl", "rkl", "fkl+rkl", "akl", "jsd"]
DATASETS = ["dolly", "self_inst", "vicuna", "sinst_11_", "uinst_11_"]
SEEDS = [10, 20, 30, 40, 50]

BASE_PATH = "results/eval_main"

def extract_rouge_l(file_path):
    """Reads the last line of the log file and extracts the rougeL score."""
    try:
        if not os.path.exists(file_path):
            return None
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
            # filtering for empty lines at the end
            lines = [l.strip() for l in lines if l.strip()]
            
            if not lines:
                return None
            
            last_line = lines[-1]
            
            # Regex to find 'rougeL': <number>
            # It looks for the specific key and captures the digits/decimals after it
            match = re.search(r"'rougeL':\s*([\d\.]+)", last_line)
            
            if match:
                return float(match.group(1))
            else:
                return None
                
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def main():
    print(f"{'Configuration':<40} | {'Seeds (Tuple)':<40} | {'Mean':<10}")
    print("-" * 85)

    for approach in APPROACHES:
        for type_val in TYPES:
            for dataset in DATASETS:
                scores = []
                
                for seed in SEEDS:
                    # Construct the path: results/eval_main/<CKPT>/<appr>/<type>/seed<seed>/<data>/log.txt
                    path = os.path.join(
                        BASE_PATH, 
                        CKPT_NAME, 
                        approach, 
                        type_val, 
                        f"seed{seed}", 
                        dataset, 
                        "log.txt"
                    )
                    
                    score = extract_rouge_l(path)
                    if score is not None:
                        scores.append(score)
                
                # If we found scores for this combination, print them
                if scores:
                    mean_val = statistics.mean(scores)
                    # Create a label for the specific combo
                    label = f"{CKPT_NAME}/{approach}/{type_val}/{dataset}"
                    # Format tuple string
                    tuple_str = "(" + ", ".join(f"{s:.3f}" for s in scores) + ")"
                    
                    print(f"{label:<40} | {tuple_str:<40} | {mean_val:.4f}")

if __name__ == "__main__":
    main()

Configuration                            | Seeds (Tuple)                            | Mean      
-------------------------------------------------------------------------------------
Qwen2.5-1.5B/distill_lens/rkl/dolly      | (27.195, 26.425, 26.553, 27.293, 26.639) | 26.8212
Qwen2.5-1.5B/distill_lens/rkl/self_inst  | (22.131, 22.015, 21.080, 21.282, 21.909) | 21.6834
Qwen2.5-1.5B/distill_lens/rkl/vicuna     | (20.417, 19.929, 21.018, 20.425, 20.992) | 20.5562
Qwen2.5-1.5B/distill_lens/rkl/sinst_11_  | (42.582, 43.101, 42.817, 42.122, 42.642) | 42.6527
Qwen2.5-1.5B/distill_lens/rkl/uinst_11_  | (38.600, 38.468, 38.746, 38.657, 38.772) | 38.6488
Qwen2.5-1.5B/kd/fkl/dolly                | (24.010, 24.210, 24.484, 25.389, 24.520) | 24.5228
Qwen2.5-1.5B/kd/fkl/self_inst            | (19.216, 20.290, 20.537, 22.511, 20.699) | 20.6505
Qwen2.5-1.5B/kd/fkl/vicuna               | (17.403, 18.602, 17.589, 17.869, 17.485) | 17.7894
Qwen2.5-1.5B/kd/fkl/sinst_11_            | (38.407, 38.011, 38.57